# 下载 Qwen3-8B 原始权重（用于 LLaMA-Factory QLoRA）

本 notebook 固定下载官方 `Qwen/Qwen3-8B` 的指定 revision，并把 Hugging Face 缓存放在 `D:\huggingface_cache`。下载内容是约 16.4 GB 的原始 `safetensors`，不是 LM Studio 使用的 GGUF。

运行前请确认右上角 kernel 为 **Python (Model_finetune)**。模型是公开仓库，不登录也能下载；登录仅用于提高 Hub 限速额度。

In [ ]:
import os
import sys
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"
CACHE_DIR = Path(r"D:\huggingface_cache")

if "model_finetune" not in sys.executable.lower():
    raise RuntimeError(f"请选择 Python (Model_finetune) kernel；当前解释器为 {sys.executable}")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

print("Python:", sys.executable)
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Cache:", CACHE_DIR)

## 可选：登录 Hugging Face

如果匿名下载出现限速，再取消下一格最后一行的注释并运行登录。不要把 token 写进 notebook。

In [ ]:
from huggingface_hub import HfApi, login

try:
    account = HfApi().whoami()
    print("已登录：", account["name"])
except Exception:
    print("当前未登录；公开模型仍可下载。遇到限速时再运行 login()。")

# login()  # 需要时取消注释，按安全提示输入 token

## 下载前确认仓库和权重大小

此步骤只读取文件清单，不下载模型权重。

In [ ]:
model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, files_metadata=True)
weight_files = sorted(
    (file.rfilename, file.size or 0)
    for file in model_info.siblings
    if file.rfilename.startswith("model-") and file.rfilename.endswith(".safetensors")
)
total_weight_bytes = sum(size for _, size in weight_files)

print("Repository:", model_info.id)
print("Resolved revision:", model_info.sha)
print("Weight shards:", len(weight_files))
print(f"Weight size: {total_weight_bytes / 1024**3:.2f} GiB")
for name, size in weight_files:
    print(f"  {name}: {size / 1024**3:.2f} GiB")

assert model_info.sha == MODEL_REVISION
assert len(weight_files) == 5

## 下载模型

下面这格会正式下载约 16.4 GB。中断后重新运行会复用已完成的分片并继续下载。

In [ ]:
from huggingface_hub import snapshot_download

snapshot_path = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=str(CACHE_DIR),
        max_workers=4,
    )
)
print("下载完成：", snapshot_path)

## 完整性检查

确认五个权重分片及训练所需配置文件都存在。

In [ ]:
required_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
]
local_weights = sorted(snapshot_path.glob("model-*.safetensors"))
missing = [name for name in required_files if not (snapshot_path / name).is_file()]
local_weight_bytes = sum(path.stat().st_size for path in local_weights)

assert len(local_weights) == 5, f"权重分片数量错误：{len(local_weights)}"
assert not missing, f"缺少文件：{missing}"
assert local_weight_bytes == total_weight_bytes, (local_weight_bytes, total_weight_bytes)

print("完整性检查通过。")
print("Snapshot:", snapshot_path)
print(f"Local weights: {local_weight_bytes / 1024**3:.2f} GiB")
print("下一步可运行 smoke test：scripts/run_cnc_qlora.ps1 -Mode smoke")

## 下一步

下载完成后先关闭 LM Studio，再在项目根目录执行：

```powershell
powershell -ExecutionPolicy Bypass -File ".\scripts\run_cnc_qlora.ps1" -Mode smoke
```

Smoke test 正常后再使用 `-Mode train` 启动正式训练。